# Лабораторная работа №4. Динамическое программирование
Борисюк Е. Г. Вариант 4.

Компания распределяет 7 одинаковых серверных модулей между тремя сервисами. Производительность сервисов складывается. Сервис обработки заказов должен получить не менее 1 модуля; ни один сервис не может получить более 3 модулей. Требуется распределить все 7 модулей так, чтобы получить максимальную суммарную производительность.

Дополнительные пакеты не нужны. Для запуска: **Среда выполнения → Выполнить все**.

[Полное пошаговое решение и диаграмма классов в PDF](https://github.com/Doney888/methods-of-optimization/blob/main/lab-04/report.pdf).

## 1. Модель
$\max Z=g_1(x_1)+g_2(x_2)+g_3(x_3)$, где $x_1+x_2+x_3=7$, $1\leq x_1\leq3$, $0\leq x_2,x_3\leq3$; все переменные целые.

| u | Заказы | Аналитика | Поиск |
|---|---:|---:|---:|
| 0 | 0 | 0 | 0 |
| 1 | 6 | 2 | 4 |
| 2 | 10 | 7 | 9 |
| 3 | 13 | 12 | 13 |
| 4 | 16 | 16 | 15 |
| 5 | 18 | 19 | 17 |
| 6 | 19 | 21 | 18 |
| 7 | 20 | 23 | 19 |

## 2. Состояния и рекурсия
$F_k(s)$ — максимум для сервисов $k,\ldots,3$ при точном распределении остатка $s$. Управление $u$ переводит состояние в $s-u$.

$$F_k(s)=\max_{u\in A_k,\ u\leq s}\{g_k(u)+F_{k+1}(s-u)\}.$$

$A_1=\{1,2,3\}$, $A_2=A_3=\{0,1,2,3\}$. Граница: $F_4(0)=0$, $F_4(s)=-\infty$ при $s>0$. Недостижимые состояния исключаются. Сначала вычисляем этапы 3, 2, 1; затем восстанавливаем планы в порядке 1, 2, 3.

В Python индексы этапов начинаются с 0. Списки `choices` хранят все управления при равенстве максимальных значений.

In [1]:
class ResourceAllocation:
    """Распределение целого ресурса между независимыми сервисами."""

    def __init__(self, total, profits, lower, upper):
        self.total = total
        self.profits = profits
        self.lower = lower
        self.upper = upper
        self.n = len(profits)
        self.values = []
        self.choices = []

    def solve(self):
        """Обратный проход: values[k][s] для сервисов k, ..., n-1."""
        self.values = [[float('-inf')] * (self.total + 1)
                       for _ in range(self.n + 1)]
        self.choices = [[[] for s in range(self.total + 1)]
                        for k in range(self.n)]
        self.values[self.n][0] = 0

        # Следующие этапы считаются раньше текущего.
        for k in range(self.n - 1, -1, -1):
            for s in range(self.total + 1):
                for u in range(self.lower[k], min(self.upper[k], s) + 1):
                    remaining = self.values[k + 1][s - u]
                    # Исключаем остатки, которые следующие сервисы не могут распределить.
                    if remaining == float('-inf'):
                        continue
                    candidate = self.profits[k][u] + remaining
                    if candidate > self.values[k][s]:
                        self.values[k][s] = candidate
                        self.choices[k][s] = [u]
                    elif candidate == self.values[k][s]:
                        # При равенстве сохраняем все оптимальные управления.
                        self.choices[k][s].append(u)
        return self.values[0][self.total]

    def restore(self, k=0, s=None):
        """Прямой проход по всем сохранённым оптимальным управлениям."""
        if s is None:
            s = self.total
        if k == self.n:
            return [()] if s == 0 else []
        plans = []
        for u in self.choices[k][s]:
            # Строим все оптимальные продолжения после текущего решения.
            for tail in self.restore(k + 1, s - u):
                plans.append((u,) + tail)
        return plans

## 3. Исходные данные
Строки исходной таблицы для $u>3$ сохранены, но ограничения `UPPER` исключают их из алгоритма.

In [2]:
TOTAL = 7
NAMES = ['Обработка заказов', 'Аналитика', 'Поиск']
PROFITS = [
    [0, 6, 10, 13, 16, 18, 19, 20],
    [0, 2, 7, 12, 16, 19, 21, 23],
    [0, 4, 9, 13, 15, 17, 18, 19],
]
LOWER = [1, 0, 0]
UPPER = [3, 3, 3]

## 4. Обратный проход и восстановление
`solve()` строит таблицы Беллмана, `restore()` возвращает все оптимальные планы. Знак `-` в выводе таблицы означает недостижимое состояние; пустой список означает отсутствие допустимого управления.

In [3]:
def show_result(problem):
    print('s:', *range(problem.total + 1))
    for k in range(problem.n - 1, -1, -1):
        row = ['-' if x == float('-inf') else str(x)
               for x in problem.values[k]]
        print(f'F{k + 1}:', ' '.join(row))
        print(f'U{k + 1}:', problem.choices[k])
    print('Максимальная производительность:', problem.values[0][problem.total])
    print('Оптимальные распределения (заказы, аналитика, поиск):')
    for plan in problem.restore():
        parts = [problem.profits[k][u] for k, u in enumerate(plan)]
        print(plan, '->', ' + '.join(map(str, parts)), '=', sum(parts))

problem = ResourceAllocation(TOTAL, PROFITS, LOWER, UPPER)
best = problem.solve()
show_result(problem)


s: 0 1 2 3 4 5 6 7
F3: 0 4 9 13 - - - -
U3: [[0], [1], [2], [3], [], [], [], []]
F2: 0 4 9 13 16 21 25 -
U2: [[0], [0], [0], [0], [2, 3], [3], [3], []]
F1: - 6 10 15 19 23 27 31
U1: [[], [1], [1, 2], [1], [1, 2], [2], [1], [1, 2]]
Максимальная производительность: 31
Оптимальные распределения (заказы, аналитика, поиск):
(1, 3, 3) -> 6 + 12 + 13 = 31
(2, 3, 2) -> 10 + 12 + 9 = 31


## 5. Пояснение расчёта и ответ
$F_2(4)=\max(2+13,7+9,12+4)=16$, управления 2 и 3.

$F_1(7)=\max(6+25,10+21,13+16)=31$, управления 1 и 2.

Восстановление: $(s,u)=(7,1)\to(6,3)\to(3,3)$ и $(7,2)\to(5,3)\to(2,2)$. В обеих ветвях конечный остаток равен нулю.

**Максимум — 31. Все оптимальные планы: (1, 3, 3) и (2, 3, 2).** Производительности: $6+12+13=31$ и $10+12+9=31$.

## 6. Независимая проверка
Перебор ниже служит только проверкой динамического программирования: сравниваем максимум и весь список оптимальных планов.

In [4]:
from itertools import product

feasible = []
for plan in product(range(4), repeat=3):
    if sum(plan) == TOTAL and plan[0] >= LOWER[0]:
        value = sum(PROFITS[k][u] for k, u in enumerate(plan))
        feasible.append((plan, value))
        print(plan, value)
check_best = max(value for plan, value in feasible)
check_plans = {plan for plan, value in feasible if value == check_best}
assert best == check_best
assert set(problem.restore()) == check_plans
print("Проверка пройдена: максимум и все оптимальные планы совпали.")


(1, 3, 3) 31
(2, 2, 3) 30
(2, 3, 2) 31
(3, 1, 3) 28
(3, 2, 2) 29
(3, 3, 1) 29
Проверка пройдена: максимум и все оптимальные планы совпали.


## 7. Вывод
Недостижимые состояния нужно исключать значением $-\infty$, чтобы не потерять требование точного распределения ресурса. Сохранение всех равных максимумов позволяет получить обе оптимальные стратегии. Результаты таблиц Беллмана и независимой проверки совпали.